# Chapter 5: DuckDB as Your Warehouse

**SQL over files • Joins across partitions • Indexes and stats • Views and macros**

This notebook demonstrates how DuckDB turns your file system into a warehouse by querying Parquet files directly, without loading data into a database.

## What You'll Learn

1. **Zero-copy queries** — query millions of rows in seconds
2. **Joins across files** — join fact tables with dimension Parquet files
3. **Partitioning for speed** — directory-based pruning
4. **Views and macros** — reusable SQL logic without a metastore
5. **Materialized summaries** — pre-aggregate for dashboard queries
6. **Portable warehouse** — a single `.duckdb` file with all metadata

## Prerequisites

```bash
pip install duckdb polars pyarrow pandas
```

## A note on data size

The chapter prose talks about the full 2023 yellow taxi corpus (≈38M rows, ≈600MB). To keep this notebook runnable on any laptop, the demo defaults to **a single month** (January 2023, ~3M rows, ~48MB). All queries and patterns are identical; only the magnitudes change. Set `MONTHS = list(range(1, 13))` in the download cell to reproduce the chapter's full-year benchmarks.

## Setup

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

print('DuckDB version:', duckdb.__version__)

# Create directories
Path('data/raw/taxi').mkdir(parents=True, exist_ok=True)
Path('data/staging/taxi').mkdir(parents=True, exist_ok=True)
print('Setup complete')

DuckDB version: 1.5.3
Setup complete


## 1. Download NYC Taxi Data

Demo default: **January 2023 only** (~48MB, ~3M rows). Change `MONTHS` to download more.

In [2]:
con = duckdb.connect()

MONTHS = [1]  # change to list(range(1, 13)) for the full year (~600MB)

for month in MONTHS:
    url = (
        f'https://d37ci6vzurychx.cloudfront.net/trip-data/'
        f'yellow_tripdata_2023-{month:02d}.parquet'
    )
    output = f'data/raw/taxi/yellow_tripdata_2023-{month:02d}.parquet'

    con.execute(f"""
        COPY (
            SELECT * FROM read_parquet('{url}')
        ) TO '{output}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print(f'Downloaded month {month}')

print(f'Total months downloaded: {len(MONTHS)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Downloaded month 1
Total months downloaded: 1


## 2. Zero-Copy Queries: Query Files Directly

The killer feature: DuckDB queries Parquet files without loading them into a database.

In [3]:
# Aggregate across every Parquet file in the directory.
# DuckDB does column projection + predicate pushdown automatically.
result = con.execute("""
    SELECT
        DATE_TRUNC('month', tpep_pickup_datetime) AS month,
        COUNT(*) AS trips,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(AVG(total_amount), 2) AS avg_fare
    FROM read_parquet('data/raw/taxi/yellow_tripdata_*.parquet')
    WHERE trip_distance > 0
      AND tpep_pickup_datetime >= TIMESTAMP '2023-01-01'
      AND tpep_pickup_datetime <  TIMESTAMP '2024-01-01'
    GROUP BY 1
    ORDER BY 1
""").df()

print('Monthly trip statistics:')
result

Monthly trip statistics:


,month,trips,avg_distance,avg_fare
0,2023-01-01,3020862,3.91,26.98
1,2023-02-01,10,4.57,31.38


### What Just Happened?

- DuckDB read the Parquet file(s) **without loading into a database**
- Used **predicate pushdown** (the `trip_distance > 0` filter and the date range)
- Scanned only the **4 columns** it needed out of 19
- Parallelized across all CPU cores automatically

### Why the date filter?

The raw TLC files contain a small number of rows with out-of-range pickup timestamps (typos in vendor uploads — you'll see a handful of 2008 and 2024 dates in the 2023 January file). Filtering to the expected calendar year keeps results clean.

## 3. Create Reference (Dimension) Tables

In [4]:
# Rate codes
con.execute("""
    COPY (
        SELECT * FROM (VALUES
            (1, 'Standard rate'),
            (2, 'JFK'),
            (3, 'Newark'),
            (4, 'Nassau or Westchester'),
            (5, 'Negotiated fare'),
            (6, 'Group ride')
        ) AS t(rate_code_id, description)
    ) TO 'data/raw/taxi/rate_codes.parquet' (FORMAT PARQUET)
""")

# Payment types
con.execute("""
    COPY (
        SELECT * FROM (VALUES
            (1, 'Credit card'),
            (2, 'Cash'),
            (3, 'No charge'),
            (4, 'Dispute'),
            (5, 'Unknown'),
            (6, 'Voided trip')
        ) AS t(payment_type_id, description)
    ) TO 'data/raw/taxi/payment_types.parquet' (FORMAT PARQUET)
""")

print('Reference tables created')

Reference tables created


## 4. Joins Across Partitions

Join the trip facts with reference dimensions — all read from Parquet.

In [5]:
# Airport trips by payment method
query = """
SELECT
    r.description AS rate_type,
    p.description AS payment_method,
    COUNT(*) AS trips,
    ROUND(AVG(t.fare_amount), 2) AS avg_fare,
    ROUND(SUM(t.fare_amount), 2) AS total_revenue
FROM read_parquet('data/raw/taxi/yellow_tripdata_*.parquet') t
JOIN read_parquet('data/raw/taxi/rate_codes.parquet') r
    ON t.RatecodeID = r.rate_code_id
JOIN read_parquet('data/raw/taxi/payment_types.parquet') p
    ON t.payment_type = p.payment_type_id
WHERE r.rate_code_id IN (2, 3)  -- JFK and Newark only
GROUP BY 1, 2
ORDER BY 3 DESC
"""

con.execute(query).df()

,rate_type,payment_method,trips,avg_fare,total_revenue
0,JFK,Credit card,90665,69.98,6345071.05
1,JFK,Cash,20757,65.85,1366938.65
2,Newark,Credit card,6265,90.08,564378.00
3,Newark,Cash,2091,68.96,144185.90
4,JFK,Dispute,1812,6.30,11417.00
5,JFK,No charge,1005,23.30,23416.75
6,Newark,Dispute,346,5.48,1894.50
7,Newark,No charge,256,26.38,6752.00


## 5. Partitioning for Speed

Repartition by month so queries that filter on time can prune entire directories.

Two small things to note:

1. We cast `DATE_TRUNC('month', ...)` to `DATE`. If you leave it as `TIMESTAMP`, DuckDB writes URL-encoded directory names like `pickup_month=2023-01-01%2000%3A00%3A00/` instead of `pickup_month=2023-01-01/`.
2. We restrict to 2023 dates so the dirty timestamps don't each get their own partition.

In [6]:
import os
print('Partitioning by month...')

con.execute("""
    COPY (
        SELECT
            *,
            CAST(DATE_TRUNC('month', tpep_pickup_datetime) AS DATE) AS pickup_month
        FROM read_parquet('data/raw/taxi/yellow_tripdata_*.parquet')
        WHERE tpep_pickup_datetime >= TIMESTAMP '2023-01-01'
          AND tpep_pickup_datetime <  TIMESTAMP '2024-01-01'
    )
    TO 'data/staging/taxi/partitioned'
    (FORMAT PARQUET, PARTITION_BY (pickup_month), OVERWRITE_OR_IGNORE)
""")

partitions = sorted(
    d for d in os.listdir('data/staging/taxi/partitioned')
    if d.startswith('pickup_month=')
)
print(f'Created {len(partitions)} partitions:')
for p in partitions:
    print(f'  - {p}/')

Partitioning by month...


Created 2 partitions:
  - pickup_month=2023-01-01/
  - pickup_month=2023-02-01/


### Query a Single Partition

When the demo runs on a single month, the speed-up is small (there's only ~3M rows to scan). With the full year it's roughly 10x.

In [7]:
import time

# Partitioned: glob exactly one directory
start = time.time()
result_partitioned = con.execute("""
    SELECT
        COUNT(*) AS trips,
        ROUND(SUM(total_amount), 2) AS revenue
    FROM read_parquet('data/staging/taxi/partitioned/pickup_month=2023-01-01/*.parquet')
""").df()
time_partitioned = time.time() - start

# Full scan with a WHERE clause
start = time.time()
result_full = con.execute("""
    SELECT
        COUNT(*) AS trips,
        ROUND(SUM(total_amount), 2) AS revenue
    FROM read_parquet('data/raw/taxi/yellow_tripdata_*.parquet')
    WHERE DATE_TRUNC('month', tpep_pickup_datetime) = TIMESTAMP '2023-01-01'
""").df()
time_full = time.time() - start

print(f'Partitioned query: {time_partitioned*1000:.0f} ms')
print(f'Full scan query:   {time_full*1000:.0f} ms')
if time_partitioned > 0:
    print(f'Speedup: {time_full / time_partitioned:.1f}x')
print()
print('January 2023:')
result_partitioned

Partitioned query: 3 ms
Full scan query:   10 ms
Speedup: 3.4x

January 2023:


,trips,revenue
0,3066718,82863379.14


## 6. Views and Macros: Reusable Logic

In [8]:
con.execute("""
    CREATE OR REPLACE VIEW clean_trips AS
    SELECT
        tpep_pickup_datetime AS pickup_time,
        tpep_dropoff_datetime AS dropoff_time,
        DATE_TRUNC('day', tpep_pickup_datetime) AS pickup_date,
        passenger_count,
        trip_distance,
        PULocationID AS pickup_zone,
        DOLocationID AS dropoff_zone,
        RatecodeID AS rate_code,
        payment_type,
        fare_amount,
        tip_amount,
        total_amount,
        -- Clean nulls and outliers
        CASE
            WHEN trip_distance <= 0 THEN NULL
            WHEN trip_distance > 100 THEN NULL
            ELSE trip_distance
        END AS clean_distance,
        CASE
            WHEN total_amount < 0 THEN NULL
            WHEN total_amount > 500 THEN NULL
            ELSE total_amount
        END AS clean_amount
    FROM read_parquet('data/staging/taxi/partitioned/**/*.parquet')
    WHERE passenger_count > 0
      AND fare_amount >= 0
""")

# Use the view
daily_revenue = con.execute("""
    SELECT
        pickup_date,
        COUNT(*) AS trips,
        ROUND(SUM(clean_amount), 2) AS revenue
    FROM clean_trips
    WHERE pickup_date BETWEEN '2023-01-01' AND '2023-01-07'
    GROUP BY 1
    ORDER BY 1
""").df()

print('Daily revenue, first week of January 2023:')
daily_revenue

Daily revenue, first week of January 2023:


,pickup_date,trips,revenue
0,2023-01-01,71203,2209648.07
1,2023-01-02,62838,1983739.78
2,2023-01-03,81615,2414929.89
3,2023-01-04,90569,2562875.31
4,2023-01-05,96307,2646976.91
5,2023-01-06,97692,2623034.06
6,2023-01-07,100248,2586784.93


### Create and Use Macros

In [9]:
con.execute("""
    CREATE OR REPLACE MACRO trip_efficiency(distance, duration_minutes) AS (
        CASE
            WHEN duration_minutes = 0 THEN NULL
            ELSE (distance / duration_minutes) * 60  -- MPH
        END
    )
""")

efficiency = con.execute("""
    SELECT
        HOUR(pickup_time) AS hour,
        ROUND(AVG(trip_efficiency(
            clean_distance,
            DATEDIFF('minute', pickup_time, dropoff_time)
        )), 2) AS avg_mph
    FROM clean_trips
    WHERE clean_distance IS NOT NULL
      AND pickup_date = '2023-01-15'
    GROUP BY 1
    ORDER BY 1
""").df()

print('Average speed by hour (Jan 15, 2023):')
efficiency

Average speed by hour (Jan 15, 2023):


,hour,avg_mph
0,0,13.51
1,1,14.45
2,2,14.57
3,3,15.67
4,4,17.48
5,5,21.92
6,6,22.15
7,7,21.04
8,8,17.96
9,9,16.27


## 7. Materialized Summaries for Dashboard Queries

In [10]:
# Pre-aggregate by date + pickup zone
con.execute("""
    COPY (
        SELECT
            pickup_date,
            pickup_zone,
            COUNT(*) AS trip_count,
            ROUND(SUM(clean_amount), 2) AS total_revenue,
            ROUND(AVG(clean_distance), 2) AS avg_distance,
            ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY clean_amount), 2)
                AS median_fare
        FROM clean_trips
        GROUP BY 1, 2
    ) TO 'data/staging/taxi/daily_summary.parquet' (FORMAT PARQUET)
""")

# Read the summary back — milliseconds, not seconds
top_zones = con.execute("""
    SELECT
        pickup_zone,
        SUM(trip_count) AS total_trips,
        ROUND(SUM(total_revenue), 2) AS revenue
    FROM read_parquet('data/staging/taxi/daily_summary.parquet')
    WHERE pickup_date BETWEEN '2023-01-01' AND '2023-01-31'
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").df()

print('Top 10 pickup zones, January 2023:')
top_zones

Top 10 pickup zones, January 2023:


,pickup_zone,total_trips,revenue
0,132,155104.0,11866128.31
1,237,141881.0,2798698.73
2,236,131629.0,2712439.17
3,161,130019.0,3070524.91
4,186,105512.0,2463702.59
5,162,101486.0,2344850.21
6,142,95603.0,2036420.11
7,230,95024.0,2496438.33
8,138,87257.0,5578655.01
9,170,84651.0,1939021.31


## 8. Real-World Query Patterns

### Time-Series Analysis

In [11]:
# Hourly demand by day of week (single month so we get every weekday)
hourly_pattern = con.execute("""
    SELECT
        DAYNAME(pickup_time) AS day_of_week,
        HOUR(pickup_time) AS hour,
        COUNT(*) AS trips
    FROM clean_trips
    GROUP BY 1, 2
    ORDER BY
        CASE DAYNAME(pickup_time)
            WHEN 'Monday'    THEN 1
            WHEN 'Tuesday'   THEN 2
            WHEN 'Wednesday' THEN 3
            WHEN 'Thursday'  THEN 4
            WHEN 'Friday'    THEN 5
            WHEN 'Saturday'  THEN 6
            WHEN 'Sunday'    THEN 7
        END,
        hour
""").df()

print(f'Rows: {len(hourly_pattern)}')
hourly_pattern.head(12)

Rows: 168


,day_of_week,hour,trips
0,Monday,0,6772
1,Monday,1,3500
2,Monday,2,1890
3,Monday,3,1436
4,Monday,4,1401
5,Monday,5,2428
6,Monday,6,6651
7,Monday,7,12348
8,Monday,8,16679
9,Monday,9,18446


### Window Functions for Ranking

In [12]:
monthly_leaders = con.execute("""
    WITH monthly_zones AS (
        SELECT
            DATE_TRUNC('month', pickup_date) AS month,
            pickup_zone,
            COUNT(*) AS trips,
            ROW_NUMBER() OVER (
                PARTITION BY DATE_TRUNC('month', pickup_date)
                ORDER BY COUNT(*) DESC
            ) AS rank
        FROM clean_trips
        GROUP BY 1, 2
    )
    SELECT month, pickup_zone, trips, rank
    FROM monthly_zones
    WHERE rank <= 5
    ORDER BY month, rank
""").df()

print('Top 5 zones per month:')
monthly_leaders

Top 5 zones per month:


,month,pickup_zone,trips,rank
0,2023-01-01,132,155104,1
1,2023-01-01,237,141881,2
2,2023-01-01,236,131629,3
3,2023-01-01,161,130019,4
4,2023-01-01,186,105512,5
5,2023-02-01,230,3,1
6,2023-02-01,137,1,2
7,2023-02-01,211,1,3
8,2023-02-01,68,1,4
9,2023-02-01,246,1,5


## 9. Build the Portable Warehouse

Save all views and macros to a `.duckdb` file so anyone can re-open and query without re-running setup.

In [13]:
warehouse_con = duckdb.connect('data/taxi_warehouse.duckdb')

warehouse_con.execute("""
    CREATE OR REPLACE VIEW trips AS
    SELECT * FROM read_parquet('data/staging/taxi/partitioned/**/*.parquet')
""")

warehouse_con.execute("""
    CREATE OR REPLACE VIEW daily_summary AS
    SELECT * FROM read_parquet('data/staging/taxi/daily_summary.parquet')
""")

warehouse_con.execute("""
    CREATE OR REPLACE VIEW clean_trips AS
    SELECT
        tpep_pickup_datetime AS pickup_time,
        tpep_dropoff_datetime AS dropoff_time,
        DATE_TRUNC('day', tpep_pickup_datetime) AS pickup_date,
        passenger_count,
        trip_distance,
        PULocationID AS pickup_zone,
        DOLocationID AS dropoff_zone,
        RatecodeID AS rate_code,
        payment_type,
        fare_amount,
        tip_amount,
        total_amount,
        CASE
            WHEN trip_distance <= 0 THEN NULL
            WHEN trip_distance > 100 THEN NULL
            ELSE trip_distance
        END AS clean_distance,
        CASE
            WHEN total_amount < 0 THEN NULL
            WHEN total_amount > 500 THEN NULL
            ELSE total_amount
        END AS clean_amount
    FROM read_parquet('data/staging/taxi/partitioned/**/*.parquet')
    WHERE passenger_count > 0
      AND fare_amount >= 0
""")

warehouse_con.execute("""
    CREATE OR REPLACE MACRO trip_efficiency(distance, duration_minutes) AS (
        CASE
            WHEN duration_minutes = 0 THEN NULL
            ELSE (distance / duration_minutes) * 60
        END
    )
""")

warehouse_con.close()

import os
size_kb = os.path.getsize('data/taxi_warehouse.duckdb') / 1024
print(f'Portable warehouse created: data/taxi_warehouse.duckdb ({size_kb:.0f} KB)')
print()
print('The .duckdb file only stores metadata, views, and macros.')
print('All data still lives in the Parquet files.')

Portable warehouse created: data/taxi_warehouse.duckdb (268 KB)

The .duckdb file only stores metadata, views, and macros.
All data still lives in the Parquet files.


### Use the Portable Warehouse

In [14]:
warehouse_con = duckdb.connect('data/taxi_warehouse.duckdb', read_only=True)

result = warehouse_con.execute('SELECT COUNT(*) FROM trips').fetchone()
print(f'Total trips in warehouse: {result[0]:,}')

revenue = warehouse_con.execute("""
    SELECT
        pickup_date,
        ROUND(SUM(clean_amount), 2) AS revenue
    FROM clean_trips
    WHERE pickup_date BETWEEN '2023-01-01' AND '2023-01-07'
    GROUP BY 1
    ORDER BY 1
""").df()

warehouse_con.close()
print()
print('First week revenue (via portable warehouse):')
revenue

Total trips in warehouse: 3,066,728

First week revenue (via portable warehouse):


,pickup_date,revenue
0,2023-01-01,2209648.07
1,2023-01-02,1983739.78
2,2023-01-03,2414929.89
3,2023-01-04,2562875.31
4,2023-01-05,2646976.91
5,2023-01-06,2623034.06
6,2023-01-07,2586784.93


## Summary

You learned how to:

1. Run **zero-copy queries** directly over Parquet files
2. **Join** fact and dimension tables that live as plain files
3. **Partition** for directory-level pruning, with `DATE`-typed keys for clean names
4. Define **views and macros** that ship inside a `.duckdb` file
5. Build **materialized summaries** for sub-second dashboard queries
6. Package everything as a **portable warehouse** — a single `.duckdb` file that points at your Parquet lake

## Performance Benchmarks

**Test machine:** M2 MacBook Pro, 16GB RAM, **full year (≈38M rows)**:

| Query Type | Rows Scanned | Execution Time |
|---|---|---|
| Full aggregation | 38M | ~3s |
| Filtered aggregation (single partition) | ~3M | ~0.3s |
| Join with reference dimensions | 38M | ~4s |
| Window function (ROW_NUMBER) | 38M | ~6s |
| Summary table query | ~70K rows | <50ms |

Running the same six queries hourly for a month on a Snowflake XS warehouse comes to ~$24. DuckDB: $0.

## Next Steps

Chapter 6 shows how to chain DuckDB with Polars for complex transformations without Spark.